In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

BRONZE = "workspace.complaints.bronze_complaints"
TARGET_PATH = "/Workspace/Users/furyboy4592@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples/incremental_load_sample.csv"
CURRENT_RUN_DATE = datetime.now(timezone.utc).date()
BATCH_ID = f"batch_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"

# 1. Read CSV using header so columns map by name (not arbitrary position)
df_raw = spark.read.option("header", "true").csv(TARGET_PATH)

# 2. Canonical column list matching the dataset
STANDARD_COLS = [
    "complaint_id", "date_received", "product", "sub_product", "issue", "sub_issue",
    "company", "state", "zip_code", "submitted_via", "date_sent_to_company",
    "company_response", "timely", "company_public_response", "tags"
]

# Ensure any missing columns exist as null strings
for c in STANDARD_COLS:
    if c not in df_raw.columns:
        df_raw = df_raw.withColumn(c, F.lit(None).cast("string"))

# Select explicitly by name and add audit metadata
df_bronze = df_raw.select([F.col(c).cast("string") for c in STANDARD_COLS]) \
    .withColumn("load_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.lit("incremental_load_sample.csv")) \
    .withColumn("batch_id", F.lit(BATCH_ID)) \
    .withColumn("load_type", F.lit("incremental")) \
    .withColumn("run_date", F.lit(CURRENT_RUN_DATE))

# 3. Safe creation and idempotent overwrite
try:
    spark.table(BRONZE)
    table_exists = True
except Exception:
    table_exists = False

if not table_exists:
    # First write: creates the Delta table partitioned by run_date
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("run_date") \
        .saveAsTable(BRONZE)
    print(f"Bronze table created: {df_bronze.count()} rows loaded (partitioned by run_date).")
else:
    # Subsequent writes: overwrites only today's partition
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"run_date = '{CURRENT_RUN_DATE}'") \
        .saveAsTable(BRONZE)
    print(f"Bronze partition refreshed: {df_bronze.count()} rows loaded for run_date={CURRENT_RUN_DATE}.")

# Signal SUCCESS to master orchestrator
dbutils.notebook.exit("SUCCESS")